# Project Canopy: Kaggle training notebook 

### Load necessary packages  

In [ ]:
%pip install rasterio -q
%pip install torchgeo -q
%pip install rioxarray -q
%pip install glob2 -q
%pip install mlflow -q
%pip install dagshub -q
%pip install lightning -q

In [ ]:
import numpy as np
import torch
import lightning as L
from lightning.pytorch.loggers import MLFlowLogger
import glob2

import torchgeo
import requests
import datetime
import os
from pathlib import Path
import dagshub

from torchgeo.datasets import RasterDataset, unbind_samples, stack_samples, IntersectionDataset
from torchgeo.samplers import RandomGeoSampler, PreChippedGeoSampler, Units
from torchgeo.transforms import indices
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp

### Link up with DagsHub to get code

In [ ]:
# Get creds
DAGSHUB_REPO_OWNER = "Omdena"
DAGSHUB_REPO_NAME = input("Enter Dagshub repo name: ") or "ProjectCanopy2"
DAGSHUB_USER_NAME = input("Enter DagsHub user name: ")
DAGSHUB_TOKEN = input("Enter DagsHub token: ")
DAGSHUB_REMOTE_PATH = input("Enter DAgshub remote path: ") or "task03_modeling"
DAGSHUB_LOCAL = input("Enter Dagshub local path: ") or "/kaggle/working/dh"

!dagshub login

In [ ]:
dh_api = dagshub.common.api.repo.RepoAPI(f"{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}")
dh_api.download(remote_path=DAGSHUB_REMOTE_PATH, local_path=DAGSHUB_LOCAL)

In [ ]:
"""
To import downloaded code, assuming os.getcwd() returns /kaggle/working:

from dh import *
"""
import dh.util.utils as utils
import dh.models.lightning_model as lm

In [ ]:
import importlib

In [ ]:
# TODO: run if code needs to be rewritten while working---may not work on Kaggle
importlib.reload(utils)
importlib.reload(lm)

### Set up MLFlow for experiment tracking 

#### MLFlow configuration

Uses [these docs](https://dagshub.com/docs/integration_guide/mlflow_tracking/).

In [ ]:

MLFLOW_EXPERIMENT_NAME = input("Please enter the MLFlow experiment name or skip to use 'default'") or "kaggle_projcanopy"
print("MLFlow experiment name: ",MLFLOW_EXPERIMENT_NAME)

In [ ]:
## Following code here: https://dagshub.com/docs/integration_guide/mlflow_tracking/
## Also here: https://colab.research.google.com/drive/1XLP2Ouxk-k6y9yOxc4Grp-Aq6aGcbhuj

# Point to DagsHub repo
# os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USER_NAME
# os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
os.environ['MLFLOW_TRACKING_URI'] = f'https://dagshub.com/{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}.mlflow'
dagshub.init(DAGSHUB_REPO_NAME, DAGSHUB_REPO_OWNER, mlflow=True)

# os.environ['MLFLOW_ARTIFACT_LOC'] = 'mlflow-artifacts:/experiments' # TODO: check this
# mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

# Create experiment and set its location
# os.environ['MLFLOW_EXPERIMENT_NAME'] = MLFLOW_EXPERIMENT_NAME
# mlflow.create_experiment(os.environ['MLFLOW_EXPERIMENT_NAME'], os.environ['MLFLOW_ARTIFACT_LOC'])
# mlflow.set_experiment(experiment_name=os.environ['MLFLOW_EXPERIMENT_NAME'])

# set up logger for Lightning: can use different experiment_name if desired
mlf_logger = MLFlowLogger(experiment_name="lightning_logs", tracking_uri=os.environ['MLFLOW_TRACKING_URI'])

## Configure and load data

### Kaggle data paths

Once you have added a Kaggle dataset to the notebook, enter the path below. It
should start with /kaggle/input/[DATASET_NAME]/. Modify as needed for your own
directory structure.

In [ ]:
# inputs
data_base_dir = None # TODO: path to dataset root folder
data_x_dir = os.path.join(data_base_dir, None) # TODO: path to original images
data_y_dir = os.path.join(data_base_dir, None) # TODO: path to original masks

# outputs
output_base_dir = None # TODO: path to all outputs, should start /kaggle/working
reprojected_dir = os.path.join(output_base_dir, "reprojected")
reprojected_x_dir = os.path.join(reprojected_dir, "x")
reprojected_y_dir = os.path.join(reprojected_dir, "y")
reprojected_split_dir = os.path.join(reprojected_dir, "split")
objects_dir = os.path.join(output_base_dir, "objects") # model checkpoints, etc

### Datasets, dataloaders, datamodules, preprocessing

#### Reproject scenes and move images

In [ ]:

# handle imagery
orig_x_names, reproj_x_names = utils.DataPreprocessing.get_and_copy_files(data_x_dir, 
                                                                    reprojected_x_dir)

# handle masks
orig_y_names, reproj_y_names = utils.DataPreprocessing.get_and_copy_files(data_y_dir,
                                                                    reprojected_y_dir)

In [ ]:
utils.DataPreprocessing.reproject_scenes_truths(orig_y_names, 
                                          reproj_y_names, 
                                          orig_x_names, 
                                          reproj_x_names)

In [ ]:
utils.DataPreprocessing.check(reproj_x_names, reproj_y_names)

#### Split

#### Create dataset

In [ ]:
dataset = utils.CanopyDataset(reprojected_x_dir, reprojected_y_dir)
train_size = int(0.8 * len(dataset))
validation_size = len(dataset) - train_size
train_dataset, validation_dataset = torch.utils.data.random_split(dataset, 
                                                            [train_size, validation_size])

In [ ]:
train_dataloader = DataLoader(train_dataset,batch_size=32, shuffle = True)
valid_dataloader = DataLoader(validation_dataset, batch_size=32, shuffle = True)

### Configure and create/load model

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# model creation
BACKBONE = 'resnet50'
channels = 12
segmodel = smp.Unet(BACKBONE, classes=2, in_channels=channels, activation='sigmoid').to(device)
preprocess_input = smp.encoders.get_preprocessing_fn(BACKBONE, pretrained='imagenet')

In [ ]:
for param in list(segmodel.encoder.parameters())[:]:
    param.requires_grad = False

In [ ]:
criterion = smp.losses.DiceLoss('binary')
metric_dict = {"loss": criterion,
               "iou": utils.custom_iou}

params_to_update = []
for name, param in segmodel.named_parameters():
    if param.requires_grad == True:
        params_to_update.append(param)

optimizer = torch.optim.Adam(params=segmodel.parameters(), lr=0.001)

#### Run training and validation    

In [ ]:
lightning_model = lm.LightningModel(segmodel,
                                 optimizer,
                                 metric_dict)

In [ ]:
MAX_EPOCHS = 20

In [ ]:
trainer = L.Trainer(logger=mlf_logger, max_epochs=MAX_EPOCHS)
trainer.fit(lightning_model,
            train_dataloader,
            valid_dataloader)

#### Save model

In [ ]:
# TODO: model checkpoints can be accessed from the /kaggle/working directory

### Predictions

In [ ]:
""" TODO: here's some documentation to get you started on using the model to predict:
https://lightning.ai/docs/pytorch/stable/deploy/production_basic.html
"""